# 01 Universe and split

**What this does.** Recipe steps 1 and 2, then the training/holdout split.
1. Candidate comps: companies headquartered in the US whose SIC2 code is among the target list's codes.
2. Keep firms whose EV/EBITDA is within 3-30. The multiple is used only for this range; the financial columns are
   split off at once and saved to `08_valuation` (never an input to 02-07).
3. Split 70/30, stratified on SIC2 (codes with fewer than 6 firms merged into "other"), seed 2026, the 10
   business-structure showcase targets forced into training. Training firms are the comp pool and the development
   targets; holdout firms are the valuation test and never enter the comp pool.

**Reads:** the company snapshot named in `data_manifest.json`. **Saved outputs:** `outputs-local/v1a/01_universe/`.

**Changelog**
- 2026-11-18: created from `v1/notebooks/comp_universe_and_facets.ipynb` (universe) and
  `v1/notebooks/comps_and_valuation.ipynb` cell 7 (split, now `split.sic2_split`). Reproduces v1 exactly.
- 2026-11-19: v1 `comp_universe_and_facets.ipynb` sections 1-2 tables ported (SIC2 codes, the companies the 3-30
  range drops, by reason only, and the SIC codes that lose companies).


In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "v1a" / "current.toml").exists())
sys.path.insert(0, str(ROOT / "v1a" / "common"))
import paths, nbtools as nb
paths.setup()
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 90); pd.set_option("display.max_rows", 80)
cur = paths.current()
checks = nb.Checks()

## 1. Current setting (`v1a/current.toml`)

In [2]:
pd.Series(cur["universe"])

snapshot                                            company_list_v3
universe_run                                       20261025T060358Z
split_run                                          20261027T204930Z
universe        outputs-local/v1a/01_universe/universe/universe.csv
split                 outputs-local/v1a/01_universe/split/split.csv
dtype: object

## 2. Inputs

In [3]:
import json
from project_data import load_manifest, snapshot_path
universe_run = json.loads((paths.path(cur["universe"]["universe"]).parent / "run.json").read_text())
split_run = json.loads((paths.path(cur["universe"]["split"]).parent / "run.json").read_text())
snapshot = snapshot_path()
print("snapshot:", snapshot.relative_to(ROOT), "| sha256 matches the universe run:", nb.sha256(snapshot) == universe_run["snapshot_sha256"])
nb.inputs([cur["universe"]["universe"], cur["universe"]["split"]])

snapshot: data-local\snapshots\company_list_v3_websites_fixed.csv | sha256 matches the universe run: True


,input,rows,sha256,status
0,outputs-local/v1a/01_universe/universe/universe.csv,533,1b492b3d2181,matches seed
1,outputs-local/v1a/01_universe/split/split.csv,533,0de7aa335563,matches seed


## 3. Results

### 3.1 From the company list to the universe

In [4]:
import universe as un
companies, financials = un.build_universe()          # financials are not shown or saved here
funnel = pd.Series({
    "companies in the snapshot": len(companies),
    "HQ in the US": int(companies["us_hq"].sum()),
    "SIC2 in the target list": int(companies["sic2_in_target_list"].sum()),
    "candidates (both)": int(companies["candidate"].sum()),
    "EV/EBITDA within 3-30": int(companies["kept_multiple"].sum()),
    "in the universe": int(companies["in_universe"].sum()),
})
display(funnel.to_frame("companies"))
companies.loc[~companies["kept_multiple"], "drop_reason"].value_counts().rename("dropped by the 3-30 range")

,companies
companies in the snapshot,560
HQ in the US,560
SIC2 in the target list,560
candidates (both),560
EV/EBITDA within 3-30,533
in the universe,533


drop_reason
above_30    20
below_3      7
Name: dropped by the 3-30 range, dtype: int64

#### SIC 2-digit codes (from v1 `comp_universe_and_facets.ipynb` section 1)

Companies per 2-digit SIC code. Where a code has very few companies, a same-SIC comp may not exist in this universe. That isn't a reason to drop anything: SIC is a check, and the facets decide what counts as the same business.

In [5]:
info, _ = un.load_list()                 # v1a: the financial table is not shown here
candidates = un.select_candidates(info)
sic2 = (candidates.groupby("target_sic2_code")
        .agg(companies=("IQ_COMPANY_ID", "size"),
             example_industries=("target_primary_sic_industry",
                                 lambda s: "; ".join(s.value_counts().index[:2])))
        .sort_values("companies", ascending=False))
print("codes with 1-2 companies:", (sic2["companies"] <= 2).sum())
sic2

codes with 1-2 companies: 8


,companies,example_industries
target_sic2_code,,
73,117,"Computer Programming, Data Processing, And Other Computer Related Services; Prepackage..."
28,52,"Pharmaceutical Preparations; Biological Products, Except Diagnostic Substances"
35,38,"Special Industry Machinery, not elsewhere classified; Farm Machinery and Equipment"
38,35,"Orthopedic, Prosthetic, and Surgical Appliances and Supplies; Electromedical and Elect..."
87,22,"Management Consulting Services; Engineering, Accounting, Research, Management, and Rel..."
79,18,Miscellaneous Amusement And Recreation; Amusement and Recreation Services
20,17,Bottled and Canned Soft Drinks and Carbonated Waters; Sugar And Confectionery Products
50,16,"Machinery, Equipment, And Supplies; Medical, Dental, and Hospital Equipment and Supplies"
37,16,Motor Vehicle Parts and Accessories; Aircraft


#### Who the 3-30 range drops, and which SIC codes lose companies (v1 section 2)

v1 listed each dropped company with its multiple; here only the reason (below 3 or above 30), since 01 shows no financial values (EV/EBITDA is read for valuation only in 08).

In [6]:
dropped = (companies.loc[~companies["kept_multiple"] & companies["candidate"],
                       ["target_company_name", "target_ciq_ticker", "target_sic4_code",
                        "target_primary_sic_industry", "drop_reason"]]
           .sort_values(["drop_reason", "target_sic4_code"]))
dropped


,target_company_name,target_ciq_ticker,target_sic4_code,target_primary_sic_industry,drop_reason
95,"Celsius Holdings, Inc.",CELH,2086,Bottled and Canned Soft Drinks and Carbonated Waters,above_30
194,"FIGS, Inc.",FIGS,2300,Apparel and Other Finished Products Made From Fabrics and Similar Materials,above_30
346,Neogen Corporation,NEOG,2834,Pharmaceutical Preparations,above_30
478,"TG Therapeutics, Inc.",TGTX,2834,Pharmaceutical Preparations,above_30
552,"Xeris Biopharma Holdings, Inc.",XERS,2834,Pharmaceutical Preparations,above_30
531,"Veracyte, Inc.",VCYT,2835,In Vitro and In Vivo Diagnostic Substances,above_30
0,"AAON, Inc.",AAON,3585,Air-Conditioning and Warm Air Heating Equipment and Commercial and Industrial Refriger...,above_30
448,"Sonos, Inc.",SONO,3651,Household Audio and Video Equipment,above_30
517,Ubiquiti Inc.,UI,3663,Radio and Television Broadcasting and Communications Equipment,above_30
249,HEICO Corporation,HEI,3724,Aircraft Engines and Engine Parts,above_30


In [7]:
universe = companies[companies["in_universe"]]
lost = (companies.groupby("target_sic2_code")
        .agg(before=("IQ_COMPANY_ID", "size"), after=("in_universe", "sum")))
lost["dropped"] = lost["before"] - lost["after"]
print(f"universe: {len(universe)} companies, {universe['target_sic2_code'].nunique()} SIC 2-digit codes "
      f"(all {companies['target_sic2_code'].nunique()} still covered: "
      f"{set(universe['target_sic2_code']) == set(companies['target_sic2_code'])})")
lost[lost["dropped"] > 0].sort_values("dropped", ascending=False)

universe: 533 companies, 43 SIC 2-digit codes (all 43 still covered: True)


,before,after,dropped
target_sic2_code,,,
28,52,46,6
73,117,113,4
38,35,32,3
87,22,20,2
36,10,8,2
13,5,3,2
35,38,37,1
23,7,6,1
20,17,16,1


### 3.2 The split

In [8]:
import split as sp
universe = pd.read_csv(paths.path(cur["universe"]["universe"]), dtype=str)
rebuilt = sp.sic2_split(universe)
print(rebuilt["sample"].value_counts().to_string())
by_stratum = pd.crosstab(rebuilt["stratum"], rebuilt["sample"]).sort_values("training", ascending=False)
by_stratum.T

sample
training    373
holdout     160


stratum,73,other,28,35,38,87,79,37,49,20,...,82,34,64,59,15,36,17,23,16,42
sample,,,,,,,,,,,,,,,,,,,,,
holdout,34,15,14,11,10,6,5,4,4,5,...,4,3,3,3,3,2,2,2,2,2
training,79,35,32,26,22,14,12,11,11,11,...,9,7,7,7,6,6,4,4,4,4


In [9]:
print("showcase targets (always training):", ", ".join(sp.SHOWCASE))
balance = pd.read_csv(paths.out("01_universe") / "split" / "balance_sector.csv", index_col=0)
balance.round(1)

showcase targets (always training): PCTY, WEN, CSL, AFG, BOC, NWSA, AZZ, BBCP, BUKS, SNFC.A


,training,holdout,difference
target_sic2_code,,,
"A Agriculture, Forestry, Fishing",0.0,0.6,-0.6
B Mining,1.6,0.0,1.6
C Construction,3.8,4.4,-0.6
D Manufacturing,35.1,34.4,0.7
"E Transportation, Communications, Utilities",7.5,7.5,0.0
F Wholesale Trade,2.9,2.5,0.4
G Retail Trade,3.5,2.5,1.0
"H Finance, Insurance, Real Estate",11.5,10.6,0.9
I Services,34.0,37.5,-3.5


## 4. Checks against the saved v1 numbers

In [10]:
saved_split = pd.read_csv(paths.path(cur["universe"]["split"]), dtype=str)
in_universe = companies[companies["in_universe"]]
checks.add("universe: same 533 companies", set(in_universe["IQ_COMPANY_ID"]) == set(universe["IQ_COMPANY_ID"]) and len(universe) == 533,
           f"rebuilt {len(in_universe)}, saved {len(universe)}")
cols = list(universe.columns)
same_values = in_universe[cols].astype(str).set_index("IQ_COMPANY_ID").sort_index().fillna("nan").equals(
    universe.astype(str).set_index("IQ_COMPANY_ID").sort_index().fillna("nan"))
checks.add("universe: saved columns identical", same_values, f"{len(cols)} columns")
checks.add("universe run.json counts", universe_run["companies"] == len(companies) and universe_run["in_universe"] == len(in_universe)
           and universe_run["dropped"] == companies.loc[~companies["kept_multiple"], "drop_reason"].value_counts().to_dict(),
           json.dumps(universe_run["dropped"]))
m = rebuilt.merge(saved_split, on="IQ_COMPANY_ID", suffixes=("", "_saved"))
checks.add("split: same sample for every company", len(m) == 533 and (m["sample"] == m["sample_saved"]).all(),
           f"{int((m['sample'] != m['sample_saved']).sum())} differences")
checks.add("split: 373 training / 160 holdout", rebuilt["sample"].value_counts().to_dict() == {"training": 373, "holdout": 160})
checks.add("split: showcase targets in training", rebuilt.loc[rebuilt["showcase"], "sample"].eq("training").all())
seeded_fin = pd.read_csv(paths.path(cur["valuation"]["financials"]), dtype={"IQ_COMPANY_ID": str})
checks.add("financials saved to 08_valuation match the snapshot", set(seeded_fin["IQ_COMPANY_ID"]) == set(financials["IQ_COMPANY_ID"])
           and len(seeded_fin) == len(financials), f"{len(seeded_fin)} companies (values not shown here)")
checks.show()

7/7 checks pass


,check,result,detail
0,universe: same 533 companies,pass,"rebuilt 533, saved 533"
1,universe: saved columns identical,pass,11 columns
2,universe run.json counts,pass,"{""above_30"": 20, ""below_3"": 7}"
3,split: same sample for every company,pass,0 differences
4,split: 373 training / 160 holdout,pass,
5,split: showcase targets in training,pass,
6,financials saved to 08_valuation match the snapshot,pass,560 companies (values not shown here)


## 5. History (v1)

- The universe is the team's 560-company list (decided 2026-10-24), after the two recipe filters: 533 companies.
- The first split (2026-10-25) was stratified on company size (facet F9); since 2026-10-27 every result uses this
  SIC2 split (`20261027T204930Z`). The size-stratified baseline is kept in `v1/notebooks/history/`.
- The 160 holdout companies were scored several times in v1 (10-28 holdout routes, 11-10 integration), so they are
  not a clean final test; see `10_final_holdout`.